# 실습 내용

- 데이터 : attrition.csv
- KNN 알고리즘으로 모델링한다.

# 1.환경 준비

In [23]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format='retina'

In [24]:
# 데이터 읽어오기

path = './../00_data/attrition.csv'
df = pd.read_csv(path)

# 2.데이터 이해

In [25]:
# 상위 몇 개 행 확인
df.head()

,Attrition,Age,DistanceFromHome,EmployeeNumber,Gender,JobSatisfaction,MaritalStatus,MonthlyIncome,OverTime,PercentSalaryHike,TotalWorkingYears
0,0,33,7,817,Male,3,Married,11691,No,11,14
1,0,35,18,1412,Male,4,Single,9362,No,11,10
2,0,42,6,1911,Male,1,Married,13348,No,13,18
3,0,46,2,1204,Female,1,Married,17048,No,23,28
4,1,22,4,593,Male,3,Single,3894,No,16,4


**데이터 설명**

- Attrition: 이직 여부 (1: 이직, 0: 잔류)
- Age: 나이
- DistanceFromHome: 집-직장 거리 (단위: 마일)
- EmployeeNumber: 사번
- Gender: 성별 (Male, Female)
- JobSatisfaction: 직무 만족도(1: Low, 2: Medium, 3: High, 4: Very High)
- MaritalStatus: 결혼 상태 (Single, Married, Divorced)
- MonthlyIncome: 월급 (단위: 달러)
- OverTime: 야근 여부 (Yes, No)
- PercentSalaryHike: 전년 대비 급여 인상율(단위: %)
- TotalWorkingYears: 총 경력 연수

In [26]:
# Target 값 분포 확인
target = 'Attrition'

print(df[target].value_counts())
print('----------')
print(df[target].value_counts(normalize=True))

Attrition
0    1001
1     195
Name: count, dtype: int64
----------
Attrition
0    0.836957
1    0.163043
Name: proportion, dtype: float64


In [27]:
# 기술통계 확인
df.describe()

,Attrition,Age,DistanceFromHome,EmployeeNumber,JobSatisfaction,MonthlyIncome,PercentSalaryHike,TotalWorkingYears
count,1196.000000,1196.00000,1196.000000,1196.000000,1196.000000,1196.000000,1196.000000,1196.000000
mean,0.163043,36.94398,9.258361,1035.629599,2.716555,6520.104515,15.251672,11.330268
std,0.369560,9.09270,8.166016,604.340130,1.110962,4665.902253,3.625946,7.823821
min,0.000000,18.00000,1.000000,1.000000,1.000000,1009.000000,11.000000,0.000000
25%,0.000000,30.00000,2.000000,507.750000,2.000000,2928.250000,12.000000,6.000000
50%,0.000000,36.00000,7.000000,1028.000000,3.000000,4973.500000,14.000000,10.000000
75%,0.000000,43.00000,14.000000,1581.250000,4.000000,8420.500000,18.000000,15.000000
max,1.000000,60.00000,29.000000,2068.000000,4.000000,19999.000000,25.000000,40.000000


In [28]:
# NaN 값 확인
df.isna().sum()


Attrition            0
Age                  0
DistanceFromHome     0
EmployeeNumber       0
Gender               0
JobSatisfaction      0
MaritalStatus        0
MonthlyIncome        0
OverTime             0
PercentSalaryHike    0
TotalWorkingYears    0
dtype: int64

In [29]:
# 상관관계 확인
df.corr(numeric_only=True)


,Attrition,Age,DistanceFromHome,EmployeeNumber,JobSatisfaction,MonthlyIncome,PercentSalaryHike,TotalWorkingYears
Attrition,1.000000,-0.167866,0.081973,-0.008707,-0.078936,-0.163572,-0.000048,-0.182162
Age,-0.167866,1.000000,-0.010917,-0.023786,-0.012425,0.490107,-0.008303,0.674331
DistanceFromHome,0.081973,-0.010917,1.000000,0.054948,-0.021623,-0.012803,0.052348,0.002606
EmployeeNumber,-0.008707,-0.023786,0.054948,1.000000,-0.022863,-0.014032,-0.009514,-0.016317
JobSatisfaction,-0.078936,-0.012425,-0.021623,-0.022863,1.000000,-0.025082,0.030811,-0.039380
MonthlyIncome,-0.163572,0.490107,-0.012803,-0.014032,-0.025082,1.000000,-0.021334,0.768437
PercentSalaryHike,-0.000048,-0.008303,0.052348,-0.009514,0.030811,-0.021334,1.000000,-0.021988
TotalWorkingYears,-0.182162,0.674331,0.002606,-0.016317,-0.039380,0.768437,-0.021988,1.000000


In [30]:
df.head()

,Attrition,Age,DistanceFromHome,EmployeeNumber,Gender,JobSatisfaction,MaritalStatus,MonthlyIncome,OverTime,PercentSalaryHike,TotalWorkingYears
0,0,33,7,817,Male,3,Married,11691,No,11,14
1,0,35,18,1412,Male,4,Single,9362,No,11,10
2,0,42,6,1911,Male,1,Married,13348,No,13,18
3,0,46,2,1204,Female,1,Married,17048,No,23,28
4,1,22,4,593,Male,3,Single,3894,No,16,4


In [31]:
# df = df.replace({})
print(df['Gender'].value_counts())
print('----')
print(df['MaritalStatus'].value_counts())
print('----')
print(df['OverTime'].value_counts())

Gender
Male      727
Female    469
Name: count, dtype: int64
----
MaritalStatus
Married     548
Single      384
Divorced    264
Name: count, dtype: int64
----
OverTime
No     854
Yes    342
Name: count, dtype: int64


# 3.데이터 전처리

**1) 변수 제거**

- 제거 대상 변수: EmployeeNumber

In [32]:
# 제거 대상: EmployeeNumber

# 변수 제거
df = df.drop('EmployeeNumber', axis=1)


# 확인
df.head()


,Attrition,Age,DistanceFromHome,Gender,JobSatisfaction,MaritalStatus,MonthlyIncome,OverTime,PercentSalaryHike,TotalWorkingYears
0,0,33,7,Male,3,Married,11691,No,11,14
1,0,35,18,Male,4,Single,9362,No,11,10
2,0,42,6,Male,1,Married,13348,No,13,18
3,0,46,2,Female,1,Married,17048,No,23,28
4,1,22,4,Male,3,Single,3894,No,16,4


**2) x, y 분리**

In [33]:
# target 확인
target = 'Attrition'

# 데이터 분리
X = df.drop(target, axis=1)
y = df[target]

**3) 가변수화**

In [44]:
X[['Gender', 'MaritalStatus', 'OverTime']] = X[['Gender', 'MaritalStatus', 'OverTime']].replace({'Male':0, 'Female':1, 'MaritalStatus':0, 'Married':1, 'Single':2, 'Divorced':3, 'No':0, 'Yes':1}).astype(int)
X.head()

,Age,DistanceFromHome,Gender,JobSatisfaction,MaritalStatus,MonthlyIncome,OverTime,PercentSalaryHike,TotalWorkingYears
0,33,7,0,3,1,11691,0,11,14
1,35,18,0,4,2,9362,0,11,10
2,42,6,0,1,1,13348,0,13,18
3,46,2,1,1,1,17048,0,23,28
4,22,4,0,3,2,3894,0,16,4


In [45]:
# X = X.replace({})
print(X['Gender'].value_counts())
print('----')
print(X['MaritalStatus'].value_counts())
print('----')
print(X['OverTime'].value_counts())

Gender
0    727
1    469
Name: count, dtype: int64
----
MaritalStatus
1    548
2    384
3    264
Name: count, dtype: int64
----
OverTime
0    854
1    342
Name: count, dtype: int64


In [46]:
# 가변수화 대상: Gender, JobSatisfaction, MaritalStatus, OverTime


# 가변수화


# 확인
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Attrition          1196 non-null   int64
 1   Age                1196 non-null   int64
 2   DistanceFromHome   1196 non-null   int64
 3   Gender             1196 non-null   int64
 4   JobSatisfaction    1196 non-null   int64
 5   MaritalStatus      1196 non-null   int64
 6   MonthlyIncome      1196 non-null   int64
 7   OverTime           1196 non-null   int64
 8   PercentSalaryHike  1196 non-null   int64
 9   TotalWorkingYears  1196 non-null   int64
dtypes: int64(10)
memory usage: 93.6 KB


**4) 학습용, 평가용 데이터 분리**

In [47]:
# 모듈 불러오기
from sklearn.model_selection import train_test_split

# 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.3,
                                                    random_state=1)


**5) 정규화**

In [48]:
# 모듈 불러오기
from sklearn.preprocessing import MinMaxScaler

# 정규화

scaler = MinMaxScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# 4.모델링

- 본격적으로 모델을 선언하고 학습하고 평가하는 과정을 진행합니다.

In [49]:
# 1단계: 불러오기
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [50]:
# 2단계: 선언하기
model = KNeighborsClassifier()

In [51]:
# 3단계: 학습하기

model.fit(X_train, y_train)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [52]:
# 4단계: 예측하기
y_pred = model.predict(X_test)

In [53]:
# 5단계: 평가하기
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print('-'*10)
print(classification_report(y_test, y_pred))



[[285  15]
 [ 47  12]]
----------
              precision    recall  f1-score   support

           0       0.86      0.95      0.90       300
           1       0.44      0.20      0.28        59

    accuracy                           0.83       359
   macro avg       0.65      0.58      0.59       359
weighted avg       0.79      0.83      0.80       359

